# 📚 Módulo 03 - A/B Testing y Experimentación

## 🎯 Objetivos

1. ✅ Entender qué es A/B testing y cuándo usarlo
2. ✅ Diseñar experimentos válidos
3. ✅ Calcular tamaño de muestra y duración
4. ✅ Analizar resultados con tests estadísticos

---

## 1️⃣ ¿Qué es A/B Testing?

**Definición**: Experimento controlado para comparar dos versiones (A vs B)

```
Usuarios
  │
  ├──── 50% → Versión A (Control)
  │              - Modelo actual
  │
  └──── 50% → Versión B (Tratamiento)
                 - Modelo nuevo

¿B es significativamente mejor que A?
```

### Ejemplos de A/B Testing en ML

* **Modelos**: XGBoost vs LightGBM
* **Features**: Con feature X vs sin feature X
* **Hiperparámetros**: Threshold 0.5 vs 0.7
* **Arquitecturas**: Red neuronal vs ensemble

---

## 2️⃣ Diseño de Experimentos

### Pasos

1. **Definir Hipótesis**
   * H0 (nula): No hay diferencia entre A y B
   * H1 (alternativa): B es mejor que A

2. **Elegir Métrica**
   * **Primaria**: CTR, conversion rate, accuracy
   * **Secundarias**: Latencia, costo, user satisfaction

3. **Calcular Tamaño de Muestra**

```python
from statsmodels.stats.power import zt_ind_solve_power

# ¿Cuántos usuarios necesito?
n = zt_ind_solve_power(
    effect_size=0.05,     # Mejora mínima detectable (5%)
    alpha=0.05,           # Nivel de significancia
    power=0.8,            # Poder estadístico
    alternative='larger'  # B > A
)
print(f"Tamaño de muestra por grupo: {n:.0f}")
```

4. **Asignación Aleatoria**

```python
import hashlib

def assign_variant(user_id):
    # Hash determinista
    hash_val = int(hashlib.md5(str(user_id).encode()).hexdigest(), 16)
    return 'A' if hash_val % 2 == 0 else 'B'
```

5. **Ejecutar Experimento**
   * Duración: Al menos 1 semana (capturar día de semana)
   * Monitoreo: Revisar métricas diariamente

6. **Analizar Resultados**

---

## 3️⃣ Análisis Estadístico

### Test de Hipótesis

#### Para Proporciones (conversion rate, CTR)

```python
from statsmodels.stats.proportion import proportions_ztest

# Datos
conversions_A = 450  # de 10,000 usuarios
conversions_B = 520  # de 10,000 usuarios

counts = [conversions_B, conversions_A]
nobs = [10000, 10000]

z_stat, p_value = proportions_ztest(counts, nobs)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value < 0.05:
    print("✅ B es significativamente mejor que A")
else:
    print("❌ No hay diferencia significativa")
```

#### Para Métricas Continuas (latencia, revenue)

```python
from scipy.stats import ttest_ind

revenue_A = [12.5, 10.3, 15.2, ...]  # lista de revenues
revenue_B = [13.1, 11.8, 16.5, ...]

t_stat, p_value = ttest_ind(revenue_B, revenue_A)

if p_value < 0.05:
    print("✅ B tiene revenue significativamente mayor")
```

### Intervalo de Confianza

```python
from scipy import stats

# Diferencia de medias con 95% CI
mean_diff = np.mean(revenue_B) - np.mean(revenue_A)
std_error = np.sqrt(np.var(revenue_A)/len(revenue_A) + np.var(revenue_B)/len(revenue_B))

ci_lower = mean_diff - 1.96 * std_error
ci_upper = mean_diff + 1.96 * std_error

print(f"Diferencia: ${mean_diff:.2f}")
print(f"95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")
```

---

## 4️⃣ Pitfalls Comunes

### ❌ **Peeking** (Mirar resultados antes de tiempo)

```python
# ❌ MAL
for day in range(1, 15):
    p_value = run_test(day)
    if p_value < 0.05:
        print("B wins!")
        break  # Parar experimento
```

**Problema**: Infla tasa de falsos positivos

✅ **Solución**: Definir duración fija de antemano

### ❌ **Multiple Testing** (Probar muchas variantes)

```python
# ❌ MAL: Probar 20 variantes sin corrección
for variant in range(20):
    if p_value < 0.05:
        print(f"Variant {variant} wins!")
```

✅ **Solución**: Bonferroni correction

```python
alpha_corrected = 0.05 / 20  # = 0.0025
```

### ❌ **Ignoring Novelty Effect**

Usuarios prefieren B solo porque es nuevo

✅ **Solución**: Correr experimento más tiempo (2-4 semanas)

### ❌ **Selection Bias**

No asignar aleatoriamente

✅ **Solución**: Hash determinístico + validar balance

---

## 5️⃣ A/B Testing para Modelos ML

### Shadow Mode

```
Usuario request
  │
  ├── Modelo A (Producción) → Respuesta al usuario
  │
  └── Modelo B (Shadow) → Log predictions (no responde)

Comparar offline: accuracy_A vs accuracy_B
```

### Canary Deployment

```
95% tráfico → Modelo A (Actual)
 5% tráfico → Modelo B (Nuevo)

Monitorear métricas. Si B está bien, aumentar a 50%
```

### Multi-Armed Bandit

Alternativa dinámica a A/B testing fijo

```python
# Thompson Sampling
for request in requests:
    # Elegir variante basándose en probabilidad de éxito
    variant = sample_from_beta_distributions()
    
    # Servir y observar reward
    reward = serve_and_observe(variant)
    
    # Actualizar creencias
    update_beta(variant, reward)
```

**Ventaja**: Minimiza tráfico a variante inferior

---

## 6️⃣ Checklist de A/B Testing

☑️ Hipótesis clara (H0 y H1)  
☑️ Métrica primaria definida  
☑️ Tamaño de muestra calculado  
☑️ Asignación aleatoria implementada  
☑️ Duración fija establecida  
☑️ Métricas guardroom (latencia, errores)  
☑️ No peeking hasta el final  
☑️ Corrección por multiple testing si aplica  
☑️ Plan de rollback si B falla  
☑️ Documentación de resultados  

---

**Universidad del Aconcagua 🇦🇷**